In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

In [13]:
def load_cnpq_bolsas(data_dir: Path, sample_size: int = 100000) -> pd.DataFrame:
    """Carrega o dataset CNPQ com download e limpeza automática."""
    import pyarrow.parquet as pq
    import gdown
    import os

    file_id = "1kZLrjRr4z1MGx7ehWQHXUcFwgsTqVOIX"

    base = data_dir / "dataset_cnpq_felps"
    base.mkdir(parents=True, exist_ok=True)

    arquivo_path = base / "br_cnpq_bolsas_microdados.parquet"

    # Se o ficheiro não existir ou tiver menos de 10MB (ou seja, for o HTML de erro), faz o download
    if not arquivo_path.exists() or os.path.getsize(arquivo_path) < 10000000:
        if arquivo_path.exists():
            os.remove(arquivo_path)
            
        print("Iniciando o download do ficheiro de 142MB usando gdown...")
        # Usa o id para evitar o bloqueio de antivírus da Google
        gdown.download(id=file_id, output=str(arquivo_path), quiet=False)
        print("Download concluído com sucesso!")

    table = pq.read_table(arquivo_path)
    if table.schema.pandas_metadata:
        table = table.replace_schema_metadata()

    df = table.to_pandas()

    colunas_remover = [
        #"fonte_recurso",
        #"plano_interno",
        "acao_ppa",
        "programa_ppa",
        #"unidade_orcamentaria",
        #"natureza_despesa",
        #"categoria_nivel",
        #"processo",
        #"beneficiario",
        #"palavra_chave",  
        ]

    df = df.drop(columns=[c for c in colunas_remover if c in df.columns], errors="ignore")

    missing_pct = df.isnull().mean()
    high_missing = missing_pct[missing_pct > 0.40].index.tolist()
    high_missing = [c for c in high_missing if c not in colunas_remover]
    if high_missing:
        df = df.drop(columns=high_missing, errors="ignore")

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].astype("string")

    if sample_size and len(df) > sample_size:
        df = df.sample(sample_size, random_state=42)



    return df

In [14]:
# load dataset 1
df1 = load_cnpq_bolsas(Path("data"), sample_size=100000000000000)
df1.head()


,ano,processo,beneficiario,linha_fomento,modalidade,chamada,programa_cnpq,grande_area_conhecimento,area_conhecimento,subarea_conhecimento,pais_origem,sigla_uf_origem,instituicao_origem,pais_destino,sigla_uf_destino,municipio_destino,sigla_instituicao_destino,sigla_instituicao_macro,instituicao_destino,valor
0,2020,305855/2017-4,LEANDRO RAMPIM,BOLSAS DE PRODUTIVIDADE EM PESQUISA E TECNOLOGIA,PQ,PQ - 2017 - CHAMADA CNPQ N º 12/2017 - BOLSAS ...,PROGRAMA BASICO DE ENGENHARIA AGRICOLA,CIÊNCIAS AGRÁRIAS,ENGENHARIA AGRÍCOLA,CONSERVAÇÃO DE SOLO E ÁGUA,BRA,PR,UNIVERSIDADE ESTADUAL DO CENTRO-OESTE,BRA,PR,GUARAPUAVA,UNICENTRO,UNICENTRO,UNIVERSIDADE ESTADUAL DO CENTRO-OESTE,13200.0
1,2011,303184/2007-8,MARCOS AURÉLIO ORTEGA,BOLSAS DE PRODUTIVIDADE EM PESQUISA E TECNOLOGIA,PQ,PRODUTIVIDADE EM PESQUISA - PQ - 2007,PROGRAMA BASICO DE ENGENHARIA AEROESPACIAL,ENGENHARIAS,ENGENHARIA AEROESPACIAL,AERODINÂMICA DE AERONAVES ESPACIAIS,BRA,SP,INSTITUTO TECNOLÓGICO DE AERONÁUTICA,BRA,SP,SÃO JOSÉ DOS CAMPOS,ITA,ITA,INSTITUTO TECNOLÓGICO DE AERONÁUTICA,4800.0
2,2019,305450/2017-4,RAIMUNDO CARLOS SILVÉRIO FREIRE,BOLSAS DE PRODUTIVIDADE EM PESQUISA E TECNOLOGIA,PQ,PQ - 2017 - CHAMADA CNPQ N º 12/2017 - BOLSAS ...,PROGRAMA BÁSICO DE MICROELETRÔNICA,OUTRA,MICROELETRÔNICA,PROJETO,BRA,PB,UNIVERSIDADE FEDERAL DE CAMPINA GRANDE,BRA,PB,CAMPINA GRANDE,UFCG,UFCG,UNIVERSIDADE FEDERAL DE CAMPINA GRANDE,33600.0
3,2009,309151/2008-2,CHERIAF MALIK,BOLSAS DE PRODUTIVIDADE EM PESQUISA E TECNOLOGIA,PQ,PRODUTIVIDADE EM PESQUISA - PQ - 2008,PROGRAMA BASICO DE ENGENHARIA CIVIL,ENGENHARIAS,ENGENHARIA CIVIL,MATERIAIS E COMPONENTES DE CONSTRUÇÃO,BRA,SC,UNIVERSIDADE FEDERAL DE SANTA CATARINA,BRA,SC,FLORIANÓPOLIS,UFSC,UFSC,UNIVERSIDADE FEDERAL DE SANTA CATARINA,9760.0
4,2014,308252/2011-0,DANIEL LÁZARO GALLINDO BORGES,BOLSAS DE PRODUTIVIDADE EM PESQUISA E TECNOLOGIA,PQ,PRODUTIVIDADE EM PESQUISA - PQ - 2011,PROGRAMA BASICO DE QUIMICA,CIÊNCIAS EXATAS E DA TERRA,QUÍMICA,MÉTODOS ÓTICOS DE ANÁLISE,BRA,SC,UNIVERSIDADE FEDERAL DE SANTA CATARINA,BRA,SC,FLORIANÓPOLIS,UFSC,UFSC,UNIVERSIDADE FEDERAL DE SANTA CATARINA,13200.0


In [16]:
import pandas as pd
import re

# =========================
# LOAD
# =========================
df2 = pd.read_parquet(
    r"C:\Users\felip\Documents\TCC2\dados\cnpqTcc.parquet"
)

df1_2020 = df1[df1["ano"] == 2020].copy()

# =========================
# RENAME
# =========================
df1_2020 = df1_2020.rename(columns={
    "ano": "ano_referencia",
    "valor": "valor_pago",
    "grande_area_conhecimento": "grande_area",
    "area_conhecimento": "area",
    "subarea_conhecimento": "subarea",
    "linha_fomento": "linha_de_fomento"
})

# =========================
# PALAVRAS QUE DEVEM FICAR MINÚSCULAS
# =========================
lower_words = {
    'e', 'de', 'da', 'do', 'das', 'dos',
    'em', 'para', 'com',
    'na', 'no', 'nas', 'nos',
    'a', 'o', 'as', 'os',
    'por', 'sem', 'sob',
    'entre', 'até', 'após',
    'ao', 'aos', 'à', 'às'
}

# =========================
# FUNÇÃO DE PADRONIZAÇÃO
# =========================
def formatar_titulo(texto):

    if pd.isna(texto):
        return texto

    # remove espaços extras
    texto = str(texto).strip()

    # title case
    texto = texto.title()

    palavras = texto.split()

    palavras_corrigidas = []

    for palavra in palavras:

        if palavra.lower() in lower_words:
            palavras_corrigidas.append(palavra.lower())
        else:
            palavras_corrigidas.append(palavra)

    texto = ' '.join(palavras_corrigidas)

    return texto

# =========================
# COLUNAS PARA FORMATAR
# =========================
colunas_texto = [
    'grande_area',
    'area',
    'subarea',
    'linha_de_fomento'
]

for col in colunas_texto:
    df1_2020[col] = df1_2020[col].apply(formatar_titulo)

# =========================
# CONCAT
# =========================
df_completo = pd.concat(
    [df2, df1_2020],
    ignore_index=True
)

# =========================
# CHECKS
# =========================
print("Tamanho do df1:", len(df1))
print("Tamanho do df2_2020:", len(df1_2020))
print("Tamanho total:", len(df_completo))

print("\n==== CASOS COM ' E ' ====\n")

for col in colunas_texto:

    qtd = df1_2020[col].str.contains(r'\sE\s', na=False).sum()

    print(f'{col}: {qtd}')

# =========================
# SAVE
# =========================
df_completo.to_parquet(
    r"C:\Users\felip\Documents\TCC2\dados\cnpqBolsasCompleto.parquet",
    compression="zstd",
    index=False
)

print("\nArquivo salvo com sucesso!")

Tamanho do df1: 2839708
Tamanho do df2_2020: 149553
Tamanho total: 3424331

==== CASOS COM ' E ' ====

grande_area: 0
area: 0
subarea: 0
linha_de_fomento: 0

Arquivo salvo com sucesso!
